In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
# List all CSV files in the 'datathon-2026-round-1' directory
csv_dir = 'datathon-2026-round-1'
csv_files = [f for f in os.listdir(csv_dir) if f.endswith('.csv')]

# Load each CSV file into a dataframe with variable name as the file name (without .csv)
for file in csv_files:
    var_name = os.path.splitext(file)[0]
    df = pd.read_csv(os.path.join(csv_dir, file))
    globals()[var_name] = df

/tmp/ipykernel_68007/3207619133.py:10: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(csv_dir, file))


In [ ]:
print(customers['acquisition_channel'].value_counts())

acquisition_channel
organic_search    36450
social_media      24448
paid_search       24285
email_campaign    14674
referral          12270
direct             9803
Name: count, dtype: int64
<StringArray>
[  'social_media', 'email_campaign', 'organic_search',       'referral',
         'direct',    'paid_search']
Length: 6, dtype: str


In [14]:
print(customers['acquisition_channel'].unique().value_counts())

social_media      1
email_campaign    1
organic_search    1
referral          1
direct            1
paid_search       1
Name: count, dtype: int64


In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

tx = pd.read_csv('../data/train_transaction.csv')
idn = pd.read_csv('../data/train_identity.csv')
df = tx.merge(idn, on='TransactionID', how='left').fillna('missing')

G = nx.Graph()

for _, row in df.head(200).iterrows():  # lấy mẫu nếu dữ liệu lớn
    tid = f"Txn:{row.TransactionID}"
    G.add_node(tid, type='transaction')
    G.add_node(f"Card1:{row.card1}", type='card')
    G.add_node(f"Email:{row.P_emaildomain}", type='email')
    G.add_edge(tid, f"Card1:{row.card1}")
    G.add_edge(tid, f"Email:{row.P_emaildomain}")

plt.figure(figsize=(12, 12))
nx.draw(G, with_labels=True, node_size=200, font_size=8)
plt.show()


### Câu chuyện 1: Nghịch lý Khuyến mãi - Bán nhiều nhưng có lãi không và có hàng để giao? (Marketing vs. Supply Chain)
- Câu chuyện này phân tích sự lệch pha giữa nỗ lực kéo traffic của Marketing và khả năng đáp ứng của Vận hành (Operations).

- Bảng dữ liệu kết nối:** `promotions` + `web_traffic` + `order_items` + `inventory` + `reviews`.

- Descriptive (Cái gì đã xảy ra): Biểu diễn sự gia tăng đột biến của lượng truy cập và doanh thu trong các ngày có chiến dịch khuyến mãi lớn
- Diagnostic (Tại sao lại xảy ra):** Dù tỷ lệ chuyển đổi tăng, nhưng hãy chú ý xem `stockout_flag` (cờ báo hết hàng) trong bảng `inventory` có bị kích hoạt không. Chuyện gì xảy ra khi khách đặt hàng nhưng hệ thống hết hàng? Kiểm tra xem điểm đánh giá (`rating`) trong `reviews` có giảm sút hay không do khách hàng phàn nàn.
- Predictive (Điều gì có khả năng xảy ra): Ước tính số lượng đơn hàng bị mất (Lost Sales) và mức độ sụt giảm uy tín nếu tình trạng đứt gãy chuỗi cung ứng này tiếp diễn trong kỳ khuyến mãi tiếp theo.
- Prescriptive (Đề xuất kinh doanh): Đề xuất số ngày dự trữ tồn kho (`days_of_supply`) an toàn cho các sản phẩm chủ lực (hero products) trước kỳ sale. Đề xuất quy tắc tự động tắt quảng cáo (traffic_source) cho các sản phẩm có `sell_through_rate` (tỷ lệ hàng đã bán) chạm ngưỡng 90% để tránh trải nghiệm xấu.


### Câu chuyện 2: Đãi cát tìm vàng - Nhận diện "Khách hàng Giá trị cao" vs "Thợ săn Khuyến mãi" (Customer Analytics)
- Chia phân khúc khách hàng để đầu tư tiền bạc 1 cách hợp lý

- Bảng dữ liệu kết nối:** `customers` + `orders` + `order_items` + `web_traffic`.

- Descriptive:Phân luồng tệp khách hàng theo `acquisition_channel` (kênh tiếp thị) và `age_group` (nhóm tuổi), xem nhóm nào mang lại nhiều đơn hàng nhất.
- Diagnostic: Tính toán tỷ lệ phần trăm đơn hàng áp dụng khuyến mãi (`promo_id` khác null). So sánh xem kênh `organic_search` hay `social_media` mang lại lượng khách hàng mua nguyên giá (full-price) cao hơn. Ai là người chỉ mua khi có mã giảm giá?
- Predictive:** Phân tích `inter-order gap` (số ngày giữa hai lần mua) để dự đoán thói quen mua sắm. Khách hàng từ kênh X sẽ rời bỏ (churn) nếu trong 90 ngày không có chiến dịch giảm giá?
- Prescriptive:** Đề xuất tái phân bổ ngân sách: Giảm tiền chạy quảng cáo cho nhóm "thợ săn khuyến mãi" (chỉ làm giảm biên lợi nhuận), tập trung ngân sách chăm sóc (loyalty program) cho tệp khách hàng mua nguyên giá hoặc khách hàng mua qua các kênh có `bounce_rate` (tỷ lệ thoát) thấp.

Câu chuyện 4: Rủi ro Dòng tiền và Hành vi Thanh toán (Fintech & Cash Flow Analytics)
Câu chuyện này tập trung vào dòng tiền thực tế và rủi ro hủy đơn, một vấn đề cực kỳ đau đầu trong thương mại điện tử ở Việt Nam.

Bảng dữ liệu kết nối: orders + payments + returns + products.

Descriptive: Thống kê tỷ trọng các phương thức thanh toán (payment_method: ví dụ COD, thẻ tín dụng, trả góp). So sánh Giá trị trung bình đơn hàng (AOV) giữa các phương thức này.

Diagnostic: Phương thức nào có tỷ lệ hủy đơn (order_status = 'cancelled') hoặc trả hàng cao nhất? Thông thường, thanh toán khi nhận hàng (COD) mang rủi ro "bùng hàng" rất cao. Thêm vào đó, hãy kiểm tra xem nhóm khách hàng mua trả góp (installments > 1) có xu hướng mua các sản phẩm ở phân khúc cao cấp (segment = 'Premium') nhiều hơn không.

Predictive: Giả lập dòng tiền thực nhận (Net Revenue) sẽ biến động ra sao nếu tỷ lệ thanh toán COD tiếp tục duy trì ở mức cao và đi kèm với tỷ lệ hoàn hàng hiện tại.

Prescriptive: Đề xuất chiến dịch dịch chuyển hành vi thanh toán: Cung cấp mã giảm giá nhỏ hoặc freeship đặc biệt để khuyến khích thanh toán trước (Credit Card/Ví điện tử) nhằm giảm thiểu rủi ro dòng tiền. Đồng thời, đề xuất mở rộng tích hợp các cổng thanh toán "Mua trước trả sau" (Buy Now Pay Later) cho các nhóm sản phẩm giá cao để kích cầu.

Câu chuyện 6: Trải nghiệm Giao hàng và Ranh giới của Sự Rời bỏ (Logistics & Customer Retention)
Khách hàng mua một món đồ đẹp nhưng nếu chờ quá lâu, họ có thể sẽ không bao giờ quay lại.

Bảng dữ liệu kết nối: geography + orders + shipments + reviews + customers.

Descriptive: Vẽ biểu đồ bản đồ (Map visual) hoặc Heatmap để xem phân bổ doanh thu theo khu vực (region, city). Hiển thị thời gian giao hàng trung bình (khoảng cách từ ship_date đến delivery_date) và phí vận chuyển (shipping_fee) theo từng vùng.

Diagnostic: Có mối tương quan nào giữa thời gian giao hàng dài/phí vận chuyển cao với những đánh giá tiêu cực (rating thấp) trong bảng reviews không? Liệu những khách hàng ở vùng bị giao hàng chậm có "inter-order gap" (khoảng thời gian giữa 2 lần mua) dài hơn bình thường, hay thậm chí là không bao giờ quay lại mua lần 2?

Predictive: Phân tích độ co giãn: Nếu rút ngắn thời gian giao hàng xuống 1 ngày ở vùng Central/South, dự kiến tỷ lệ khách hàng quay lại (Retention Rate) sẽ tăng thêm bao nhiêu %?

Prescriptive: Dựa trên chi phí vận chuyển và thời gian giao hàng, đề xuất ban giám đốc cân nhắc mở thêm một Hub trung chuyển (Fulfillment Center) tại khu vực có mật độ đơn hàng cao nhưng thời gian giao hàng đang bị nghẽn. Nếu chưa đủ ngân sách mở kho, cần đàm phán lại SLA (cam kết dịch vụ) với đơn vị vận chuyển thứ ba (3PL) cho các tuyến đường trọng điểm này.